# Tarea 2: Fundamentos de Python
## Ciencia de Datos Ambientales - UTEC

**Nombre:** [Rocio Trinidad Mantari Huarcaya]  
**Puntaje total:** 20 puntos

**Instrucciones:**
- Completa todos los problemas en este notebook
- Escribe tu codigo en las celdas proporcionadas
- Ejecuta todas las celdas antes de entregar
- Sube el archivo `.ipynb` completado al modulo correspondiente en Canvas

**Integridad academica:** Tarea individual. Puedes consultar materiales del curso y documentacion de Python, pero todo el codigo debe ser tuyo.

---

## Problema 1: Procesador de Nombres de Archivos Landsat (10 puntos)

Trabajas con imagenes satelitales Landsat del Peru. Los nombres siguen el formato:

```
LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF
```
Componentes: `{sensor}_{nivel}_{path_row}_{fecha}_{coleccion}_{tier}_SR_{banda}.TIF`

> Los paths 003-009, filas 062-071 cubren el territorio peruano (Madre de Dios, Loreto, Lima, Cusco).

### Tus tareas:

**Parte A (4 pts):** Funcion `procesar_nombre_landsat(nombre_archivo)` que devuelva un diccionario con:
- `sensor` (ej. "LC08"), `path` (ej. "008"), `row` (ej. "067")
- `fecha` formateada como "AAAA-MM-DD"
- `banda` (ej. "B4")

**Parte B (3 pts):** Funcion `clasificar_banda(banda)` que devuelva el nombre segun la tabla:

| Banda | Nombre |
|-------|--------|
| B1 | Aerosol costero | B2 | Azul | B3 | Verde | B4 | Rojo |
| B5 | Infrarrojo cercano (NIR) | B6 | SWIR1 | B7 | SWIR2 |

Si no esta en la tabla, devuelve "Desconocida".

**Parte C (3 pts):** Procesa la lista de archivos: parsea, imprime resumen (fecha/path/row/banda) y cuenta cuantas fechas unicas hay.

In [2]:
# Archivos Landsat sobre el Peru (paths 008-009: Madre de Dios, Ucayali, Loreto)
archivos = [
    "LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF",
    "LC08_L2SP_008067_20240615_02_T1_SR_B5.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B3.TIF",
    "LC08_L2SP_009067_20240615_02_T1_SR_B4.TIF",
    "LC09_L2SP_008067_20240708_02_T1_SR_B6.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B4.TIF",
]

# Parte A: funcion procesar_nombre_landsat

def procesar_nombre_landsat(nombre_archivo):
    partes = nombre_archivo.split("_")

    sensor = partes[0]
    path = partes[2][:3]
    row = partes[2][3:]
    fecha = partes[3]
    banda = partes[7].replace(".TIF", "")

    fecha = fecha[:4] + "-" + fecha[4:6] + "-" + fecha[6:]

    return {
        "sensor": sensor,
        "path": path,
        "row": row,
        "fecha": fecha,
        "banda": banda
    }


# Parte B: funcion clasificar_banda

def clasificar_banda(banda):
    if banda == "B1":
        return "Aerosol costero"
    elif banda == "B2":
        return "Azul"
    elif banda == "B3":
        return "Verde"
    elif banda == "B4":
        return "Rojo"
    elif banda == "B5":
        return "Infrarrojo cercano (NIR)"
    elif banda == "B6":
        return "SWIR1"
    elif banda == "B7":
        return "SWIR2"
    else:
        return "Desconocida"


# Parte C: procesar todos los archivos

fechas = []

for archivo in archivos:
    datos = procesar_nombre_landsat(archivo)
    banda = clasificar_banda(datos["banda"])

    print(datos["sensor"], datos["path"], datos["row"],
          datos["fecha"], datos["banda"], banda)

    if datos["fecha"] not in fechas:
        fechas.append(datos["fecha"])

print("Fechas únicas:", fechas)
print("Cantidad de fechas únicas:", len(fechas))

LC08 008 067 2024-06-15 B4 Rojo
LC08 008 067 2024-06-15 B5 Infrarrojo cercano (NIR)
LC08 008 067 2024-07-01 B3 Verde
LC08 009 067 2024-06-15 B4 Rojo
LC09 008 067 2024-07-08 B6 SWIR1
LC08 008 067 2024-07-01 B4 Rojo
Fechas únicas: ['2024-06-15', '2024-07-01', '2024-07-08']
Cantidad de fechas únicas: 3


---
## Problema 2: Inventario Forestal en Madre de Dios (10 puntos)

El **SERFOR** realiza inventarios forestales en Madre de Dios. Los datos incluyen valores faltantes (`-999`) y mediciones con posibles errores.

**Parte A (3 pts):** Funcion `calcular_area_basal(dap_cm)`:
- Devuelve AB en m2: $AB = \pi 	\times  (DAP/200)^2$
- Devuelve `None` si DAP <= 0 o == -999

**Parte B (3 pts):** Funcion `clasificar_arbol(dap_cm, altura_m)` que devuelva:
- `clase`: "Brinzal" (<10cm), "Latizal" (10-25cm), "Fustal menor" (25-50cm), "Fustal mayor" (>=50cm)
- `alerta`: True si DAP > 200cm, altura > 60m, o altura < 1m con DAP > 10cm

**Parte C (4 pts):** Procesa los datos:
1. Para cada árbol, calcule el área basal y clasifíquelo.
2. Omita los árboles con datos faltantes (valores -999).
3. Imprima una advertencia para los árboles marcados.
4. Calcule e imprima las estadísticas descriptivas:
- Número total de árboles válidos
- Área basal total (suma de todos los árboles válidos)
- Cantidad de árboles en cada clase de tamaño
- Número de registros marcados

In [3]:
# Inventario forestal - Madre de Dios, Peru (datos SERFOR)
# Formato: [id, especie, dap_cm, altura_m]
datos_arboles = [
    [1,  "Swietenia macrophylla",      35.4, 22.1],   # Caoba
    [2,  "Cedrela odorata",            28.2, 18.5],   # Cedro
    [3,  "Cedrelinga cateniformis",   -999,  25.0],   # Tornillo - DAP faltante
    [4,  "Virola surinamensis",        18.7, 12.3],   # Cumala
    [5,  "Dipteryx micrantha",         52.1, 24.8],   # Shihuahuaco
    [6,  "Calycophyllum spruceanum",    8.5,  6.2],   # Capirona
    [7,  "Terminalia oblonga",         45.0, 85.0],   # Yacushapana - altura sospechosa
    [8,  "Cedrelinga cateniformis",    62.3, 28.4],   # Tornillo
    [9,  "Swietenia macrophylla",      41.2, -999],   # Caoba - altura faltante
    [10, "Hura crepitans",             22.5,  0.5],   # Catahua - sospechoso
    [11, "Schizolobium parahybum",      5.2,  3.1],   # Pino chuncho
    [12, "Guazuma crinita",            38.9, 21.7],   # Bolaina
]

# Parte A: Escribe la función calcular_area_basal aqui:

def calcular_area_basal(dap_cm):
    if dap_cm == -999 or dap_cm <= 0:
        return None

    area_basal = 3.1416 * (dap_cm / 200) ** 2
    return area_basal


# Parte B: Escribe la función clasificar_arbol aquí:

def clasificar_arbol(dap_cm, altura_m):
    if dap_cm < 10:
        clase = "Brinzal"
    elif dap_cm < 25:
        clase = "Latizal"
    elif dap_cm < 50:
        clase = "Fustal menor"
    else:
        clase = "Fustal mayor"

    alerta = False

    if dap_cm > 200:
        alerta = True

    if altura_m > 60:
        alerta = True

    if altura_m < 1 and dap_cm > 10:
        alerta = True

    return clase, alerta


# Parte C: Procesar los datos e imprimir los resultados

total_validos = 0
area_total = 0

brinzal = 0
latizal = 0
fustal_menor = 0
fustal_mayor = 0

registros_marcados = 0

for arbol in datos_arboles:
    id_arbol = arbol[0]
    especie = arbol[1]
    dap = arbol[2]
    altura = arbol[3]

    if dap == -999 or altura == -999:
        continue

    area = calcular_area_basal(dap)
    clase, alerta = clasificar_arbol(dap, altura)

    total_validos += 1
    area_total += area

    if clase == "Brinzal":
        brinzal += 1
    elif clase == "Latizal":
        latizal += 1
    elif clase == "Fustal menor":
        fustal_menor += 1
    else:
        fustal_mayor += 1

    if alerta:
        registros_marcados += 1

    print(id_arbol, especie, clase, round(area, 4))

print("Numero total de arboles validos:", total_validos)
print("Area basal total:", round(area_total, 4))
print("Brinzal:", brinzal)
print("Latizal:", latizal)
print("Fustal menor:", fustal_menor)
print("Fustal mayor:", fustal_mayor)
print("Registros marcados:", registros_marcados)

1 Swietenia macrophylla Fustal menor 0.0984
2 Cedrela odorata Fustal menor 0.0625
4 Virola surinamensis Latizal 0.0275
5 Dipteryx micrantha Fustal mayor 0.2132
6 Calycophyllum spruceanum Brinzal 0.0057
7 Terminalia oblonga Fustal menor 0.159
8 Cedrelinga cateniformis Fustal mayor 0.3048
10 Hura crepitans Latizal 0.0398
11 Schizolobium parahybum Brinzal 0.0021
12 Guazuma crinita Fustal menor 0.1188
Numero total de arboles validos: 10
Area basal total: 1.0318
Brinzal: 2
Latizal: 2
Fustal menor: 4
Fustal mayor: 2
Registros marcados: 2


---
## Lista de verificacion
- [ ] Todas las celdas corren sin errores
- [ ] Ambos problemas estan completos
- [ ] Salidas visibles en todas las celdas
- [ ] Nombre incluido